# Final question set - manual answers

This notebook manually computes the answers the full task set so that the gold tool paths can be verified. Under each task template, we organise the questions in the following order:
- answerable questions on the Easy dataset (E1 and E2)
- answerable questions on the Hard dataset (H1 and H2, sometimes H3)
- abstention questions on the Hard dataset (H3)

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import textwrap
from statsmodels.stats.proportion import proportions_ztest

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 4)

In [3]:
df_questions = pd.read_csv("../benchmark_outputs/v2/question_set_v2_params.csv")
df_questions = df_questions[["task_id", "template", "dataset", "answerable", "question"]]
df_questions.head()

,task_id,template,dataset,answerable,question
0,T1-E1,T1- Single value retrieval,easy,yes,What proportion of Banking reviews that mentio...
1,T1-E2,T1- Single value retrieval,easy,yes,What proportion of Fashion reviews that mentio...
2,T1-H1,T1- Single value retrieval,hard,yes,What proportion of Fashion reviews that mentio...
3,T1-H2,T1- Single value retrieval,hard,yes,What proportion of Price Comparison reviews th...
4,T1-H3,T1- Single value retrieval,hard,no (abstain),What proportion of Ride Hailing reviews that m...


In [4]:
df_easy = pd.read_csv("../benchmark_outputs/v2/easy/labels.csv")
df_easy.head()

,review_id,industry,org,aspect,sentiment
0,R000001,Consulting,Castellan Partners,attitude-of-staff,negative
1,R000002,Price Comparison,PricePilot,price-value-for-money,negative
2,R000003,Price Comparison,CompareHive,account-access,negative
3,R000004,Consulting,Castellan Partners,attitude-of-staff,positive
4,R000005,Consulting,Merrow & Pike,discounts-promotions,positive


In [5]:
df_hard = pd.read_csv("../benchmark_outputs/v2/hard/labels.csv")
df_hard.head()

,review_id,industry,org,aspect,sentiment
0,R000001,Fashion,Northerly,general-satisfaction,positive
1,R000002,Trading,Quantly,ease-of-use,positive
2,R000003,Fashion,Northerly,attitude-of-staff,negative
3,R000004,Price Comparison,CompareHive,app-website,negative
4,R000005,Banking,Halden Savings,ease-of-use,negative


### Conventions

- **Topic** = `aspect` (the 12 leaf aspects). 
- **Negative sentiment rate** = negative reviews / *all* reviews mentioning that slice, i.e. neutral reviews stay in the denominator.
- **Complaint rate** for an organisation = its negative reviews / all its reviews.
- **Minimum volume** = 30 mentions, applied wherever the question asks for it.
- **Abstention rule**: abstain when the slice the question names has no rows at all, or has
  fewer than 30 reviews, or cannot support the shape of the answer asked for
  (e.g. a "top 3" where fewer than 3 topics exist). T5 is the exception to the 30-review floor:
  a two-proportion z-test is judged on the four cells it actually consumes, so each side needs at
  least 10 negative *and* 10 non-negative reviews on the topic, however large the two slices are.
  Every abstention answer opens with `No answer.` followed by the reason.
- **Non-actionable topics**: `competitor`, `general-satisfaction` and `reviews` describe how customers
  feel overall rather than something an organisation can fix. They are excluded from the prescriptive
  recommendations in the final answers of T8-T10.
- **Topic wording**: answers name topics in plain English, the printed tables keep the raw labels:

  | label | in the answer | label | in the answer |
  | --- | --- | --- | --- |
  | `account-access` | account access | `email` | email support |
  | `app-website` | the app or website | `general-satisfaction` | general satisfaction |
  | `attitude-of-staff` | staff attitude | `phone` | phone support |
  | `competitor` | comparisons with competitors | `price-value-for-money` | price and value for money |
  | `discounts-promotions` | discounts and promotions | `reviews` | other reviews and forums |
  | `ease-of-use` | ease of use | `speed` | speed |

- **Answer wording**: every cell computes its own evidence from scratch and then prints the answer
  it supports. The phrasing follows the "high-quality answer" examples in the task-set design
  document and is kept consistent within a template, but it is written out per question rather than
  produced by a shared builder, so the wording can follow what the numbers actually show.
  Percentages are reported to one decimal place; p-values as `p < 0.001` or to three decimals.
- **Statistical testing** = two-sided two-proportion z-test with pooled variance, significance at alpha = 0.05.


In [6]:
MIN_N = 30   # minimum mentions: used by the templates that name it, and as the
             # data-sufficiency floor for deciding abstention

# Aspects that describe how customers feel overall rather than something the organisation
# can fix. Dropped from the prescriptive recommendation (T8-T10); kept in descriptive and
# diagnostic analysis (T1-T7).
NON_ACTIONABLE = ["competitor", "general-satisfaction", "reviews"]

def ask(task_id):
    """Print the question and return the right dataset (Easy or Hard)."""
    r = df_questions.loc[df_questions.task_id == task_id].iloc[0]
    print(f"{r.task_id}  |  dataset: {r.dataset}  |  answerable: {r.answerable}")
    print(textwrap.fill(r.question, width=150))
    print("-" * 100)
    return df_easy if r.dataset == "easy" else df_hard

# Plain-English topic names for the answers; the printed tables keep the raw labels.
# Answers interpolate from this rather than spelling topics out, so a mis-typed topic
# name cannot disagree with the table above it.
ASPECT = {
    "account-access": "account access",
    "app-website": "the app or website",
    "attitude-of-staff": "staff attitude",
    "competitor": "comparisons with competitors",
    "discounts-promotions": "discounts and promotions",
    "ease-of-use": "ease of use",
    "email": "email support",
    "general-satisfaction": "general satisfaction",
    "phone": "phone support",
    "price-value-for-money": "price and value for money",
    "reviews": "other reviews and forums",
    "speed": "speed",
}


## T1- Single value retrieval

In [6]:
df = ask("T1-E1")

segment = df[(df.industry == "Banking") & (df.aspect == "app-website")]
total_count = len(segment) 
neg_count = (segment["sentiment"] == "negative").sum()
proportion = neg_count / total_count

print(f"\nANSWER: {proportion:.1%} of Banking reviews about the app or website are negative "
      f"({neg_count} out of {total_count}).")

T1-E1  |  dataset: easy  |  answerable: yes
What proportion of Banking reviews that mention the app or website are negative?
----------------------------------------------------------------------------------------------------

ANSWER: 18.7% of Banking reviews about the app or website are negative (80 out of 427).


In [7]:
80/427

0.1873536299765808

In [8]:
df = ask("T1-E2")

segment = df[(df.industry == "Fashion") & (df.aspect == "ease-of-use")]
total_count = len(segment) 
pos_count = (segment["sentiment"] == "positive").sum()
proportion = pos_count / total_count

print(f"\nANSWER: {proportion:.1%} of Fashion reviews about ease of use are positive "
      f"({pos_count} out of {total_count}).")

T1-E2  |  dataset: easy  |  answerable: yes
What proportion of Fashion reviews that mention ease of use are positive?
----------------------------------------------------------------------------------------------------

ANSWER: 86.8% of Fashion reviews about ease of use are positive (402 out of 463).


In [9]:
df = ask("T1-H1")

segment = df[(df.industry == "Fashion") & (df.aspect == "speed")]
total_count = len(segment) 
neg_count = (segment["sentiment"] == "negative").sum()
proportion = neg_count / total_count

print(f"\nANSWER: {proportion:.1%} of Fashion reviews about speed are negative "
      f"({neg_count} out of {total_count}).")

T1-H1  |  dataset: hard  |  answerable: yes
What proportion of Fashion reviews that mention speed are negative?
----------------------------------------------------------------------------------------------------

ANSWER: 21.3% of Fashion reviews about speed are negative (117 out of 549).


In [10]:
df = ask("T1-H2")

segment = df[(df.industry == "Price Comparison") & (df.aspect == "attitude-of-staff")]
total_count = len(segment) 
neg_count = (segment["sentiment"] == "negative").sum()
proportion = neg_count / total_count

print(f"\nANSWER: {proportion:.1%} of Price Comparison reviews about staff attitude are negative "
      f"({neg_count} out of {total_count}).")

T1-H2  |  dataset: hard  |  answerable: yes
What proportion of Price Comparison reviews that mention staff attitude are negative?
----------------------------------------------------------------------------------------------------

ANSWER: 10.9% of Price Comparison reviews about staff attitude are negative (43 out of 396).


In [11]:
# abstention: not enough reviews
df = ask("T1-H3")

segment = df[(df.industry == "Ride Hailing") & (df.aspect == "discounts-promotions")]
total_count = len(segment) 
neg_count = (segment["sentiment"] == "negative").sum()
proportion = neg_count / total_count

print(f"\nWRONG ANSWER: {proportion:.1%} of Ride Hailing reviews about discounts and promotions are negative "
      f"({neg_count} out of {total_count}).")

print(f"\nCORRECT ANSWER: No answer. Ride Hailing has only {total_count} reviews mentioning discounts and")
print(f"promotions (minimum {MIN_N}), so no proportion can be reported reliably.")

T1-H3  |  dataset: hard  |  answerable: no (abstain)
What proportion of Ride Hailing reviews that mention discounts and promotions are negative?
----------------------------------------------------------------------------------------------------

WRONG ANSWER: 66.7% of Ride Hailing reviews about discounts and promotions are negative (2 out of 3).

CORRECT ANSWER: No answer. Ride Hailing has only 3 reviews mentioning discounts and
promotions (minimum 30), so no proportion can be reported reliably.


## T2 - Distribution

In [12]:
df = ask("T2-E1")

segment = df[df.industry == "Ride Hailing"]
proportions = segment["sentiment"].value_counts(normalize=True).round(4)
print(proportions)

print(f"\nANSWER: The customer base is mostly dissatisfied. Ride Hailing sentiment is "
      f"{proportions['positive']:.1%} positive, {proportions['negative']:.1%} negative and "
      f"{proportions['neutral']:.1%} neutral across {len(segment)} mentions.")

T2-E1  |  dataset: easy  |  answerable: yes
What is the sentiment breakdown across all Ride Hailing reviews?
----------------------------------------------------------------------------------------------------
sentiment
negative    0.7278
positive    0.2206
neutral     0.0516
Name: proportion, dtype: float64

ANSWER: The customer base is mostly dissatisfied. Ride Hailing sentiment is 22.1% positive, 72.8% negative and 5.2% neutral across 1532 mentions.


In [13]:
0.7278+0.2206+0.0516

1.0

In [7]:
df = ask("T2-E2")

segment = df[df.org == "Marbrook"]
proportions = segment["sentiment"].value_counts(normalize=True).round(4)
print(proportions)

print(f"\nANSWER: The customer base is mostly satisfied. Marbrook sentiment is "
      f"{proportions['positive']:.1%} positive, {proportions['negative']:.1%} negative and "
      f"{proportions['neutral']:.1%} neutral across {len(segment)} mentions.")

T2-E2  |  dataset: easy  |  answerable: yes
What is the sentiment breakdown across all Marbrook reviews?
----------------------------------------------------------------------------------------------------
sentiment
positive    0.6561
negative    0.3276
neutral     0.0163
Name: proportion, dtype: float64

ANSWER: The customer base is mostly satisfied. Marbrook sentiment is 65.6% positive, 32.8% negative and 1.6% neutral across 1044 mentions.


In [8]:
df = ask("T2-H1")

segment = df[df.org == "Vanter Financial"]
proportions = segment["sentiment"].value_counts(normalize=True).round(4)
print(proportions)

print(f"\nANSWER: The customer base is mostly satisfied. Vanter Financial sentiment is "
      f"{proportions['positive']:.1%} positive, {proportions['negative']:.1%} negative and "
      f"{proportions['neutral']:.1%} neutral across {len(segment)} mentions.")

T2-H1  |  dataset: hard  |  answerable: yes
What is the sentiment breakdown across all Vanter Financial reviews?
----------------------------------------------------------------------------------------------------
sentiment
positive    0.6024
negative    0.2833
neutral     0.1143
Name: proportion, dtype: float64

ANSWER: The customer base is mostly satisfied. Vanter Financial sentiment is 60.2% positive, 28.3% negative and 11.4% neutral across 420 mentions.


In [9]:
df = ask("T2-H2")

segment = df[df.org == "PricePilot"]
proportions = segment["sentiment"].value_counts(normalize=True).round(4)
print(proportions)

print(f"\nANSWER: The customer base is mostly satisfied. PricePilot sentiment is "
      f"{proportions['positive']:.1%} positive, {proportions['negative']:.1%} negative and "
      f"{proportions['neutral']:.1%} neutral across {len(segment)} mentions.")

T2-H2  |  dataset: hard  |  answerable: yes
What is the sentiment breakdown across all PricePilot reviews?
----------------------------------------------------------------------------------------------------
sentiment
positive    0.8410
negative    0.1553
neutral     0.0037
Name: proportion, dtype: float64

ANSWER: The customer base is mostly satisfied. PricePilot sentiment is 84.1% positive, 15.5% negative and 0.4% neutral across 805 mentions.


In [10]:
df = ask("T2-H3")

segment = df[df.org == "Sable Row"]
proportions = segment["sentiment"].value_counts(normalize=True).round(4)
print(proportions)

print(f"\nANSWER: The customer base is mostly satisfied. Sable Row sentiment is "
      f"{proportions['positive']:.1%} positive, {proportions['negative']:.1%} negative and "
      f"{proportions['neutral']:.1%} neutral across {len(segment)} mentions.")

T2-H3  |  dataset: hard  |  answerable: yes
What is the sentiment breakdown across all Sable Row reviews?
----------------------------------------------------------------------------------------------------
sentiment
positive    0.6951
negative    0.3016
neutral     0.0032
Name: proportion, dtype: float64

ANSWER: The customer base is mostly satisfied. Sable Row sentiment is 69.5% positive, 30.2% negative and 0.3% neutral across 925 mentions.


## T3 - Ranking by volume

In [18]:
df = ask("T3-E1")

segment = df[(df.industry == "Travel Booking") & (df.sentiment == "negative")]
top3 = segment.aspect.value_counts().head(3)
print(top3)

print(f"\nANSWER: The top 3 topics with the most complaints in Travel Booking are: "
      f"{ASPECT[top3.index[0]]} ({top3.values[0]}), {ASPECT[top3.index[1]]} ({top3.values[1]}) "
      f"and {ASPECT[top3.index[2]]} ({top3.values[2]}).")

T3-E1  |  dataset: easy  |  answerable: yes
What are the top 3 topics with the most number of complaints in Travel Booking?
----------------------------------------------------------------------------------------------------
aspect
app-website       215
account-access    149
email             138
Name: count, dtype: int64

ANSWER: The top 3 topics with the most complaints in Travel Booking are: the app or website (215), account access (149) and email support (138).


In [11]:
df = ask("T3-E2")

segment = df[(df.industry == "Groceries") & (df.sentiment == "negative")]
top3 = segment.aspect.value_counts().head(3)
print(top3)

print(f"\nANSWER: The top 3 topics with the most complaints in Groceries are: "
      f"{ASPECT[top3.index[0]]} ({top3.values[0]}), {ASPECT[top3.index[1]]} ({top3.values[1]}) "
      f"and {ASPECT[top3.index[2]]} ({top3.values[2]}).")

T3-E2  |  dataset: easy  |  answerable: yes
What are the top 3 topics with the most number of complaints in Groceries?
----------------------------------------------------------------------------------------------------
aspect
app-website    383
ease-of-use    215
email          100
Name: count, dtype: int64

ANSWER: The top 3 topics with the most complaints in Groceries are: the app or website (383), ease of use (215) and email support (100).


In [12]:
df = ask("T3-H1")

segment = df[(df.industry == "Fashion") & (df.sentiment == "negative")]
top3 = segment.aspect.value_counts().head(3)
print(top3)

print(f"\nANSWER: The top 3 topics with the most complaints in Fashion are: "
      f"{ASPECT[top3.index[0]]} ({top3.values[0]}), {ASPECT[top3.index[1]]} ({top3.values[1]}) "
      f"and {ASPECT[top3.index[2]]} ({top3.values[2]}).")

T3-H1  |  dataset: hard  |  answerable: yes
What are the top 3 topics with the most number of complaints in Fashion?
----------------------------------------------------------------------------------------------------
aspect
app-website             241
general-satisfaction    160
speed                   117
Name: count, dtype: int64

ANSWER: The top 3 topics with the most complaints in Fashion are: the app or website (241), general satisfaction (160) and speed (117).


In [13]:
df = ask("T3-H2")

segment = df[(df.industry == "Price Comparison") & (df.sentiment == "negative")]
top3 = segment.aspect.value_counts().head(3)
print(top3)

print(f"\nANSWER: The top 3 topics with the most complaints in Price Comparison are: "
      f"{ASPECT[top3.index[0]]} ({top3.values[0]}), {ASPECT[top3.index[1]]} ({top3.values[1]}) "
      f"and {ASPECT[top3.index[2]]} ({top3.values[2]}).")

T3-H2  |  dataset: hard  |  answerable: yes
What are the top 3 topics with the most number of complaints in Price Comparison?
----------------------------------------------------------------------------------------------------
aspect
app-website              148
price-value-for-money     65
general-satisfaction      48
Name: count, dtype: int64

ANSWER: The top 3 topics with the most complaints in Price Comparison are: the app or website (148), price and value for money (65) and general satisfaction (48).


In [14]:
# abstention: not enough topics
df = ask("T3-H3")

segment = df[(df.industry == "Consulting") & (df.sentiment == "negative")]
top3 = segment.aspect.value_counts().head(3)
print(top3)

print(f"\nWRONG ANSWER: The top 3 topics with the most complaints in Consulting are: "
      f"{ASPECT[top3.index[0]]} ({top3.values[0]}).")
# This would make the user think that the answer is incomplete due to some technical error.

print("\nCORRECT ANSWER: No answer. Consulting reviews in the dataset only cover one")
print("topic (account access), so a top 3 ranking of complaint topics does not exist.")
# This highlights to the user that there are problems in the data.

T3-H3  |  dataset: hard  |  answerable: no (abstain)
What are the top 3 topics with the most number of complaints in Consulting?
----------------------------------------------------------------------------------------------------
aspect
account-access    52
Name: count, dtype: int64

WRONG ANSWER: The top 3 topics with the most complaints in Consulting are: account access (52).

CORRECT ANSWER: No answer. Consulting reviews in the dataset only cover one
topic (account access), so a top 3 ranking of complaint topics does not exist.


## T4 - Cross-segment comparison

In [23]:
df = ask("T4-E1")

segment = df[(df.aspect == "price-value-for-money")
         & (df.industry.isin(["Banking", "Ride Hailing"]))]
summary = segment.groupby("industry").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
print(summary)

neg_rate_bank = summary.loc["Banking", "neg_rate"]
neg_rate_ride = summary.loc["Ride Hailing", "neg_rate"]

print(f"\nANSWER: Ride Hailing has the higher negative sentiment rate ({neg_rate_ride:.1%}) "
      f"for price and value for money, compared to {neg_rate_bank:.1%} for Banking.")

T4-E1  |  dataset: easy  |  answerable: yes
Looking only at reviews about price and value for money, is the negative sentiment rate higher in Banking or Ride Hailing?
----------------------------------------------------------------------------------------------------
              total_count  neg_count  neg_rate
industry                                      
Banking               160         32    0.2000
Ride Hailing          120         91    0.7583

ANSWER: Ride Hailing has the higher negative sentiment rate (75.8%) for price and value for money, compared to 20.0% for Banking.


In [24]:
df = ask("T4-E2")

segment = df[(df.aspect == "app-website")
         & (df.industry.isin(["Fashion", "Travel Booking"]))]
summary = segment.groupby("industry").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
print(summary)

neg_rate_fashion = summary.loc["Fashion", "neg_rate"]
neg_rate_travel = summary.loc["Travel Booking", "neg_rate"]

print(f"\nANSWER: Travel Booking has the higher negative sentiment rate ({neg_rate_travel:.1%}) "
      f"for the app or website, compared to {neg_rate_fashion:.1%} for Fashion.")

T4-E2  |  dataset: easy  |  answerable: yes
Looking only at reviews about the app or website, is the negative sentiment rate higher in Fashion or Travel Booking?
----------------------------------------------------------------------------------------------------
                total_count  neg_count  neg_rate
industry                                        
Fashion                 788        220    0.2792
Travel Booking          488        215    0.4406

ANSWER: Travel Booking has the higher negative sentiment rate (44.1%) for the app or website, compared to 27.9% for Fashion.


In [25]:
df = ask("T4-H1")

segment = df[(df.aspect == "ease-of-use")
         & (df.industry.isin(["Groceries", "Trading"]))]
summary = segment.groupby("industry").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
print(summary)

neg_rate_groceries = summary.loc["Groceries", "neg_rate"]
neg_rate_trading = summary.loc["Trading", "neg_rate"]

print(f"\nANSWER: Groceries has the higher negative sentiment rate ({neg_rate_groceries:.1%}) "
      f"for ease of use, compared to {neg_rate_trading:.1%} for Trading.")

T4-H1  |  dataset: hard  |  answerable: yes
Looking only at reviews about ease of use, is the negative sentiment rate higher in Groceries or Trading?
----------------------------------------------------------------------------------------------------
           total_count  neg_count  neg_rate
industry                                   
Groceries          523        222    0.4245
Trading            267         29    0.1086

ANSWER: Groceries has the higher negative sentiment rate (42.4%) for ease of use, compared to 10.9% for Trading.


In [26]:
# abstention: 1 industry has no reviews
df = ask("T4-H2")

segment = df[(df.aspect == "ease-of-use")
         & (df.industry.isin(["Banking", "Consulting"]))]
summary = segment.groupby("industry").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary = summary.reindex(["Banking", "Consulting"])  # an industry with no reviews has no row to group
print(summary)

neg_rate_bank = summary.loc["Banking", "neg_rate"]

print(f"\nWRONG ANSWER: Banking has the higher negative sentiment rate ({neg_rate_bank:.1%}) "
      f"for ease of use, compared to 0% for Consulting.")

print("\nCORRECT ANSWER: No answer. Consulting has no reviews mentioning ease of use, so the two industries cannot be compared on this topic.")

T4-H2  |  dataset: hard  |  answerable: no (abstain)
Looking only at reviews about ease of use, is the negative sentiment rate higher in Banking or Consulting?
----------------------------------------------------------------------------------------------------
            total_count  neg_count  neg_rate
industry                                    
Banking           383.0       96.0    0.2507
Consulting          NaN        NaN       NaN

WRONG ANSWER: Banking has the higher negative sentiment rate (25.1%) for ease of use, compared to 0% for Consulting.

CORRECT ANSWER: No answer. Consulting has no reviews mentioning ease of use, so the two industries cannot be compared on this topic.


In [27]:
# abstention: both industries have no reviews
df = ask("T4-H3")

segment = df[(df.aspect == "price-value-for-money")
         & (df.industry.isin(["Consulting", "Streaming"]))]
summary = segment.groupby("industry").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
# the filter matches no rows at all, so there is no group to divide and no negative rate to compute
print(f"Reviews about price and value for money in Consulting or Streaming: {len(segment)}")
print(summary)

print("\nWRONG ANSWER: Consulting and Streaming have the same negative sentiment rate (0%) about price and value for money.")

print("\nCORRECT ANSWER: No answer. Neither Consulting nor Streaming has any reviews mentioning price and")
print("value for money, so the two industries cannot be compared on this topic.")

T4-H3  |  dataset: hard  |  answerable: no (abstain)
Looking only at reviews about price and value for money, is the negative sentiment rate higher in Consulting or Streaming?
----------------------------------------------------------------------------------------------------
Reviews about price and value for money in Consulting or Streaming: 0
Empty DataFrame
Columns: [total_count, neg_count]
Index: []

WRONG ANSWER: Consulting and Streaming have the same negative sentiment rate (0%) about price and value for money.

CORRECT ANSWER: No answer. Neither Consulting nor Streaming has any reviews mentioning price and
value for money, so the two industries cannot be compared on this topic.


## T5 - Statistical testing

- For each organisation, we count the negative and non-negative reviews and calculate negative sentiment rates.
- A two-proportion z-test is run only when both groups have more than 10 negative and more than 10 non-negative reviews, ensuring sufficient data for the test.
- If the data threshold is met, the negative sentiment rates are compared using a two-sided test at the 5% significance level; otherwise, the analysis reports insufficient data and provides no statistical conclusion.

In [28]:
df = ask("T5-E1")

seg_a = df[(df["org"] == "CompareHive") &    (df["aspect"] == "attitude-of-staff")]
seg_b = df[(df["org"] == "Tallywise") &    (df["aspect"] == "attitude-of-staff")]

# Counts
total_a = len(seg_a)
negative_a = (seg_a["sentiment"] == "negative").sum()
non_negative_a = total_a - negative_a

total_b = len(seg_b)
negative_b = (seg_b["sentiment"] == "negative").sum()
non_negative_b = total_b - negative_b

# Rates
rate_a = negative_a / total_a if total_a else None
rate_b = negative_b / total_b if total_b else None

print(f"CompareHive: {total_a} mentions, "    f"{negative_a} negative, {rate_a:.1%} negative sentiment rate")
print(f"Tallywise: {total_b} mentions, "    f"{negative_b} negative, {rate_b:.1%} negative sentiment rate")

# Check all four cells
sufficient_data = all([negative_a > 10, non_negative_a > 10, negative_b > 10, non_negative_b > 10])

if not sufficient_data:
    print("\nANSWER: No answer. There is insufficient data to run a reliable two-proportion z-test.")
else:
    z_stat, p_value = proportions_ztest(count=[negative_a, negative_b],nobs=[total_a, total_b],alternative="two-sided")
    print(f"\nz = {z_stat:.3f}, p = {p_value:.4f}")
    if p_value < 0.05:
        conclusion = (f"Yes. The negative sentiment rates are statistically significantly different between CompareHive ({rate_a:.1%}) and Tallywise ({rate_b:.1%}) (p = {p_value:.3f}).")
    else:
        conclusion = (f"No. The difference in negative sentiment rates between CompareHive ({rate_a:.1%}) and Tallywise ({rate_b:.1%}) is not statistically significant (p = {p_value:.3f}).")

    print("\nANSWER:", textwrap.fill(conclusion, width=150))

T5-E1  |  dataset: easy  |  answerable: yes
Do CompareHive and Tallywise have significantly different negative sentiment rates for staff attitude?
----------------------------------------------------------------------------------------------------
CompareHive: 103 mentions, 11 negative, 10.7% negative sentiment rate
Tallywise: 118 mentions, 18 negative, 15.3% negative sentiment rate

z = -1.005, p = 0.3150

ANSWER: No. The difference in negative sentiment rates between CompareHive (10.7%) and Tallywise (15.3%) is not statistically significant (p = 0.315).


In [29]:
df = ask("T5-E2")

seg_a = df[(df.org == "Trippa") & (df.aspect == "general-satisfaction")]
seg_b = df[(df.org == "Roamly") & (df.aspect == "general-satisfaction")]

# Counts
total_a = len(seg_a)
negative_a = (seg_a["sentiment"] == "negative").sum()
non_negative_a = total_a - negative_a

total_b = len(seg_b)
negative_b = (seg_b["sentiment"] == "negative").sum()
non_negative_b = total_b - negative_b

# Rates
rate_a = negative_a / total_a if total_a else None
rate_b = negative_b / total_b if total_b else None

print(f"Trippa: {total_a} mentions, "    f"{negative_a} negative, {rate_a:.1%} negative sentiment rate")
print(f"Roamly: {total_b} mentions, "    f"{negative_b} negative, {rate_b:.1%} negative sentiment rate")

# Check all four cells
sufficient_data = all([negative_a > 10, non_negative_a > 10, negative_b > 10, non_negative_b > 10])

if not sufficient_data:
    print("\nANSWER: No answer. There is insufficient data to run a reliable two-proportion z-test.")
else:
    z_stat, p_value = proportions_ztest(count=[negative_a, negative_b],nobs=[total_a, total_b],alternative="two-sided")
    print(f"\nz = {z_stat:.3f}, p = {p_value:.4f}")
    if p_value < 0.05:
        conclusion = (f"Yes. The negative sentiment rates are statistically significantly different between Trippa ({rate_a:.1%}) and Roamly ({rate_b:.1%}) (p = {p_value:.3f}).")
    else:
        conclusion = (f"No. The difference in negative sentiment rates between Trippa ({rate_a:.1%}) and Roamly ({rate_b:.1%}) is not statistically significant (p = {p_value:.3f}).")

    print("\nANSWER:", textwrap.fill(conclusion, width=150))

T5-E2  |  dataset: easy  |  answerable: yes
Do Trippa and Roamly have significantly different negative sentiment rates for general satisfaction?
----------------------------------------------------------------------------------------------------
Trippa: 123 mentions, 54 negative, 43.9% negative sentiment rate
Roamly: 95 mentions, 21 negative, 22.1% negative sentiment rate

z = 3.359, p = 0.0008

ANSWER: Yes. The negative sentiment rates are statistically significantly different between Trippa (43.9%) and Roamly (22.1%) (p = 0.001).


In [30]:
df = ask("T5-H1")

seg_a = df[(df.industry == "Groceries") & (df.aspect == "app-website")]
seg_b = df[(df.industry == "Price Comparison") & (df.aspect == "app-website")]

# Counts
total_a = len(seg_a)
negative_a = (seg_a["sentiment"] == "negative").sum()
non_negative_a = total_a - negative_a

total_b = len(seg_b)
negative_b = (seg_b["sentiment"] == "negative").sum()
non_negative_b = total_b - negative_b

# Rates
rate_a = negative_a / total_a if total_a else None
rate_b = negative_b / total_b if total_b else None

print(f"Groceries: {total_a} mentions, "    f"{negative_a} negative, {rate_a:.1%} negative sentiment rate")
print(f"Price Comparison: {total_b} mentions, "    f"{negative_b} negative, {rate_b:.1%} negative sentiment rate")

# Check all four cells
sufficient_data = all([negative_a > 10, non_negative_a > 10, negative_b > 10, non_negative_b > 10])

if not sufficient_data:
    print("\nANSWER: No answer. There is insufficient data to run a reliable two-proportion z-test.")
else:
    z_stat, p_value = proportions_ztest(count=[negative_a, negative_b],nobs=[total_a, total_b],alternative="two-sided")
    print(f"\nz = {z_stat:.3f}, p = {p_value:.4f}")
    if p_value < 0.05:
        conclusion = (f"Yes. The negative sentiment rates are statistically significantly different between Groceries ({rate_a:.1%}) and Price Comparison ({rate_b:.1%}) (p = {p_value:.3f}).")
    else:
        conclusion = (f"No. The difference in negative sentiment rates between Groceries ({rate_a:.1%}) and Price Comparison ({rate_b:.1%}) is not statistically significant (p = {p_value:.3f}).")

    print("\nANSWER:", textwrap.fill(conclusion, width=150))

T5-H1  |  dataset: hard  |  answerable: yes
Do Groceries and Price Comparison have significantly different negative sentiment rates for the app or website?
----------------------------------------------------------------------------------------------------
Groceries: 451 mentions, 217 negative, 48.1% negative sentiment rate
Price Comparison: 312 mentions, 148 negative, 47.4% negative sentiment rate

z = 0.185, p = 0.8535

ANSWER: No. The difference in negative sentiment rates between Groceries (48.1%) and Price Comparison (47.4%) is not statistically significant (p = 0.853).


In [31]:
# abstention: insufficient data
df = ask("T5-H2")

seg_a = df[(df.org == "Halden Savings") & (df.aspect == "price-value-for-money")]
seg_b = df[(df.org == "Kestrel Bank") & (df.aspect == "price-value-for-money")]

# Counts
total_a = len(seg_a)
negative_a = (seg_a["sentiment"] == "negative").sum()
non_negative_a = total_a - negative_a

total_b = len(seg_b)
negative_b = (seg_b["sentiment"] == "negative").sum()
non_negative_b = total_b - negative_b

# Rates
rate_a = negative_a / total_a if total_a else None
rate_b = negative_b / total_b if total_b else None

print(f"Halden Savings: {total_a} mentions, "    f"{negative_a} negative, {rate_a:.1%} negative sentiment rate")
print(f"Kestrel Bank: {total_b} mentions, "    f"{negative_b} negative, {rate_b:.1%} negative sentiment rate")

# Check all four cells
sufficient_data = all([negative_a > 10, non_negative_a > 10, negative_b > 10, non_negative_b > 10])

if not sufficient_data:
    print("\nANSWER: No answer. There is insufficient data to run a reliable two-proportion z-test.")
else:
    z_stat, p_value = proportions_ztest(count=[negative_a, negative_b],nobs=[total_a, total_b],alternative="two-sided")
    print(f"\nz = {z_stat:.3f}, p = {p_value:.4f}")
    if p_value < 0.05:
        conclusion = (f"Yes. The negative sentiment rates are statistically significantly different between Halden Savings ({rate_a:.1%}) and Kestrel Bank ({rate_b:.1%}) (p = {p_value:.3f}).")
    else:
        conclusion = (f"No. The difference in negative sentiment rates between Halden Savings ({rate_a:.1%}) and Kestrel Bank ({rate_b:.1%}) is not statistically significant (p = {p_value:.3f}).")

    print("\nANSWER:", textwrap.fill(conclusion, width=150))

T5-H2  |  dataset: hard  |  answerable: no (abstain)
Do Halden Savings and Kestrel Bank have significantly different negative sentiment rates for price and value for money?
----------------------------------------------------------------------------------------------------
Halden Savings: 29 mentions, 16 negative, 55.2% negative sentiment rate
Kestrel Bank: 20 mentions, 6 negative, 30.0% negative sentiment rate

ANSWER: No answer. There is insufficient data to run a reliable two-proportion z-test.


In [32]:
# abstention: insufficient data
df = ask("T5-H3")

seg_a = df[(df.org == "Northpeak Trading") & (df.aspect == "speed")]
seg_b = df[(df.org == "Quantly") & (df.aspect == "speed")]

# Counts
total_a = len(seg_a)
negative_a = (seg_a["sentiment"] == "negative").sum()
non_negative_a = total_a - negative_a

total_b = len(seg_b)
negative_b = (seg_b["sentiment"] == "negative").sum()
non_negative_b = total_b - negative_b

# Rates
rate_a = negative_a / total_a if total_a else None
rate_b = negative_b / total_b if total_b else None

print(f"Northpeak Trading: {total_a} mentions, "    f"{negative_a} negative, {rate_a:.1%} negative sentiment rate")
print(f"Quantly: {total_b} mentions, "    f"{negative_b} negative, {rate_b:.1%} negative sentiment rate")

# Check all four cells
sufficient_data = all([negative_a > 10, non_negative_a > 10, negative_b > 10, non_negative_b > 10])

if not sufficient_data:
    print("\nANSWER: No answer. There is insufficient data to run a reliable two-proportion z-test.")
else:
    z_stat, p_value = proportions_ztest(count=[negative_a, negative_b],nobs=[total_a, total_b],alternative="two-sided")
    print(f"\nz = {z_stat:.3f}, p = {p_value:.4f}")
    if p_value < 0.05:
        conclusion = (f"Yes. The negative sentiment rates are statistically significantly different between Northpeak Trading ({rate_a:.1%}) and Quantly ({rate_b:.1%}) (p = {p_value:.3f}).")
    else:
        conclusion = (f"No. The difference in negative sentiment rates between Northpeak Trading ({rate_a:.1%}) and Quantly ({rate_b:.1%}) is not statistically significant (p = {p_value:.3f}).")

    print("\nANSWER:", textwrap.fill(conclusion, width=150))

T5-H3  |  dataset: hard  |  answerable: no (abstain)
Do Northpeak Trading and Quantly have significantly different negative sentiment rates for speed?
----------------------------------------------------------------------------------------------------
Northpeak Trading: 20 mentions, 12 negative, 60.0% negative sentiment rate
Quantly: 25 mentions, 13 negative, 52.0% negative sentiment rate

ANSWER: No answer. There is insufficient data to run a reliable two-proportion z-test.


## T6 - Organisation vs industry comparison

In [33]:
df = ask("T6-E1")

segment = df[(df.industry == "Banking") & (df.aspect == "app-website")]
total_count = len(segment) 
neg_count = (segment["sentiment"] == "negative").sum()
industry_rate = neg_count / total_count
print(f"Industry rate: {industry_rate:.1%}")
print()

summary = segment.groupby("org").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop orgs with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary["vs_industry"] = summary["neg_rate"] - industry_rate
summary = summary.sort_values(by="neg_rate", ascending=False)
print(summary)
print()

answer = (f"ANSWER: The Banking industry has an average negative sentiment rate of {industry_rate:.1%} for reviews about the app or website. "
          f"Among organisations with at least {MIN_N} mentions, {summary.index[0]} has the highest negative sentiment rate at "
          f"{summary.neg_rate.iloc[0]:.1%} and is above the industry average, followed by {summary.index[1]} at {summary.neg_rate.iloc[1]:.1%}, also slightly above average. "
          f""
          f"{summary.index[2]} ({summary.neg_rate.iloc[2]:.1%}) and {summary.index[3]} ({summary.neg_rate.iloc[3]:.1%}) are below the industry average.")
print(textwrap.fill(answer, width=150))

T6-E1  |  dataset: easy  |  answerable: yes
How do individual organisations in Banking compare to the industry average on the app or website? Give me each organisation's negative sentiment rate
and ignore any organisations with less than 30 mentions about the app or website.
----------------------------------------------------------------------------------------------------
Industry rate: 18.7%

                  total_count  neg_count  neg_rate  vs_industry
org                                                            
Halden Savings             62         24    0.3871       0.1997
Northeast Bank            140         28    0.2000       0.0126
Vanter Financial           86         11    0.1279      -0.0594
Kestrel Bank              139         17    0.1223      -0.0651

ANSWER: The Banking industry has an average negative sentiment rate of 18.7% for reviews about the app or website. Among organisations with at least
30 mentions, Halden Savings has the highest negative sentiment rate

In [34]:
df = ask("T6-E2")

segment = df[(df.industry == "Travel Booking") & (df.aspect == "general-satisfaction")]
total_count = len(segment) 
neg_count = (segment["sentiment"] == "negative").sum()
industry_rate = neg_count / total_count
print(f"Industry rate: {industry_rate:.1%}")
print()

summary = segment.groupby("org").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop orgs with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary["vs_industry"] = summary["neg_rate"] - industry_rate
summary = summary.sort_values(by="neg_rate", ascending=False)
print(summary)
print()

answer = (f"ANSWER: The Travel Booking industry has an average negative sentiment rate of {industry_rate:.1%} for general satisfaction reviews. "
          f"Among organisations with at least {MIN_N} mentions, {summary.index[0]} has the highest negative sentiment rate at "
          f"{summary.neg_rate.iloc[0]:.1%} and is above the industry average, while {summary.index[1]} ({summary.neg_rate.iloc[1]:.1%}), "
          f""
          f"{summary.index[2]} ({summary.neg_rate.iloc[2]:.1%}) and {summary.index[3]} ({summary.neg_rate.iloc[3]:.1%}) are all below average.")
print(textwrap.fill(answer, width=150))

T6-E2  |  dataset: easy  |  answerable: yes
How do individual organisations in Travel Booking compare to the industry average on general satisfaction? Give me each organisation's negative
sentiment rate and ignore any organisations with less than 30 mentions about general satisfaction.
----------------------------------------------------------------------------------------------------
Industry rate: 25.5%

         total_count  neg_count  neg_rate  vs_industry
org                                                   
Trippa           123         54    0.4390       0.1841
Roamly            95         21    0.2211      -0.0339
Journeo          107         21    0.1963      -0.0587
Wayfare           79          7    0.0886      -0.1663

ANSWER: The Travel Booking industry has an average negative sentiment rate of 25.5% for general satisfaction reviews. Among organisations with at
least 30 mentions, Trippa has the highest negative sentiment rate at 43.9% and is above the industry average, whi

In [35]:
df = ask("T6-H1")

segment = df[(df.industry == "Fashion") & (df.aspect == "app-website")]
total_count = len(segment) 
neg_count = (segment["sentiment"] == "negative").sum()
industry_rate = neg_count / total_count
print(f"Industry rate: {industry_rate:.1%}")
print()

summary = segment.groupby("org").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop orgs with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary["vs_industry"] = summary["neg_rate"] - industry_rate
summary = summary.sort_values(by="neg_rate", ascending=False)
print(summary)
print()

answer = (f"ANSWER: The Fashion industry has an average negative sentiment rate of {industry_rate:.1%} for reviews about the app or website. "
          f"Among organisations with at least {MIN_N} mentions, {summary.index[0]} has the highest negative sentiment rate at "
          f"{summary.neg_rate.iloc[0]:.1%} and is above the industry average, followed by {summary.index[1]} at {summary.neg_rate.iloc[1]:.1%}, also above average. "
          f""
          f"{summary.index[2]} ({summary.neg_rate.iloc[2]:.1%}) and {summary.index[3]} ({summary.neg_rate.iloc[3]:.1%}) are below average.")
print(textwrap.fill(answer, width=150))

T6-H1  |  dataset: hard  |  answerable: yes
How do individual organisations in Fashion compare to the industry average on the app or website? Give me each organisation's negative sentiment rate
and ignore any organisations with less than 30 mentions about the app or website.
----------------------------------------------------------------------------------------------------
Industry rate: 27.5%

            total_count  neg_count  neg_rate  vs_industry
org                                                      
Marbrook            276        103    0.3732       0.0984
Northerly           192         63    0.3281       0.0533
Vella & Co          156         29    0.1859      -0.0889
Sable Row           253         46    0.1818      -0.0930

ANSWER: The Fashion industry has an average negative sentiment rate of 27.5% for reviews about the app or website. Among organisations with at least
30 mentions, Marbrook has the highest negative sentiment rate at 37.3% and is above the industry averag

In [36]:
# abstention: the industry has no reviews on this topic
df = ask("T6-H2")

segment = df[(df.industry == "Consulting") & (df.aspect == "app-website")]
total_count = len(segment) 
print(f"Total mentions: {total_count}")

print("\nANSWER: No answer. No organisation in Consulting has any reviews about the app or website, so no comparison can be made.")

T6-H2  |  dataset: hard  |  answerable: no (abstain)
How do individual organisations in Consulting compare to the industry average on the app or website? Give me each organisation's negative sentiment
rate and ignore any organisations with less than 30 mentions about the app or website.
----------------------------------------------------------------------------------------------------
Total mentions: 0

ANSWER: No answer. No organisation in Consulting has any reviews about the app or website, so no comparison can be made.


In [37]:
# abstention: the industry has no reviews on this topic
df = ask("T6-H3")

segment = df[(df.industry == "Streaming") & (df.aspect == "general-satisfaction")]
total_count = len(segment) 
print(f"Total mentions: {total_count}")

print("\nANSWER: No answer. No organisation in Streaming has any reviews about general satisfaction, so no comparison can be made.")

T6-H3  |  dataset: hard  |  answerable: no (abstain)
How do individual organisations in Streaming compare to the industry average on general satisfaction? Give me each organisation's negative sentiment
rate and ignore any organisations with less than 30 mentions about general satisfaction.
----------------------------------------------------------------------------------------------------
Total mentions: 0

ANSWER: No answer. No organisation in Streaming has any reviews about general satisfaction, so no comparison can be made.


## T7 - Driver identification by severity

In [38]:
df = ask("T7-E1")

segment = df[df.org == "Northeast Bank"]
summary = segment.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop orgs with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary = summary.sort_values(by="neg_rate", ascending=False)
print(summary)
print()
answer = (f"ANSWER: {ASPECT[summary.index[0]].capitalize()} is Northeast Bank's biggest driver of dissatisfaction. It has the "
          f"highest negative sentiment rate at {summary.neg_rate.iloc[0]:.1%} among all topics with at least {MIN_N} mentions, "
          f"ahead of {ASPECT[summary.index[1]]} ({summary.neg_rate.iloc[1]:.1%}) and {ASPECT[summary.index[2]]} "
          f"({summary.neg_rate.iloc[2]:.1%}).")
print(textwrap.fill(answer, width=150))

T7-E1  |  dataset: easy  |  answerable: yes
What is the biggest driver of dissatisfaction at Northeast Bank? Rank topics by negative sentiment rate and ignore any topics with less than 30
mentions.
----------------------------------------------------------------------------------------------------
                       total_count  neg_count  neg_rate
aspect                                                 
email                           40         35    0.8750
discounts-promotions            40         33    0.8250
attitude-of-staff               40         31    0.7750
account-access                  40         25    0.6250
reviews                         40         25    0.6250
phone                           40         21    0.5250
ease-of-use                     57         20    0.3509
competitor                      40         10    0.2500
app-website                    140         28    0.2000
speed                           40          6    0.1500
general-satisfaction         

In [39]:
df = ask("T7-E2")

segment = df[df.org == "Wayfare"]
summary = segment.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop orgs with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary = summary.sort_values(by="neg_rate", ascending=False)
print(summary)
print()
answer = (f"ANSWER: {ASPECT[summary.index[0]].capitalize()} is Wayfare's biggest driver of dissatisfaction. It has the "
          f"highest negative sentiment rate at {summary.neg_rate.iloc[0]:.1%} among all topics with at least {MIN_N} mentions, "
          f"ahead of {ASPECT[summary.index[1]]} ({summary.neg_rate.iloc[1]:.1%}) and {ASPECT[summary.index[2]]} "
          f"({summary.neg_rate.iloc[2]:.1%}).")
print(textwrap.fill(answer, width=150))

T7-E2  |  dataset: easy  |  answerable: yes
What is the biggest driver of dissatisfaction at Wayfare? Rank topics by negative sentiment rate and ignore any topics with less than 30 mentions.
----------------------------------------------------------------------------------------------------
                       total_count  neg_count  neg_rate
aspect                                                 
account-access                  40         37    0.9250
email                           40         34    0.8500
phone                           40         27    0.6750
attitude-of-staff               40         24    0.6000
discounts-promotions            40         22    0.5500
app-website                    159         82    0.5157
reviews                         40         15    0.3750
ease-of-use                     46         16    0.3478
speed                           40          8    0.2000
competitor                      40          7    0.1750
price-value-for-money           40  

In [40]:
df = ask("T7-H1")

segment = df[df.org == "Sable Row"]
summary = segment.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop orgs with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary = summary.sort_values(by="neg_rate", ascending=False)
print(summary)
print()
answer = (f"ANSWER: {ASPECT[summary.index[0]].capitalize()} is Sable Row's biggest driver of dissatisfaction. It has the "
          f"highest negative sentiment rate at {summary.neg_rate.iloc[0]:.1%} among all topics with at least {MIN_N} mentions, "
          f"ahead of {ASPECT[summary.index[1]]} ({summary.neg_rate.iloc[1]:.1%}) and {ASPECT[summary.index[2]]} "
          f"({summary.neg_rate.iloc[2]:.1%}).")
print(textwrap.fill(answer, width=150))

T7-H1  |  dataset: hard  |  answerable: yes
What is the biggest driver of dissatisfaction at Sable Row? Rank topics by negative sentiment rate and ignore any topics with less than 30 mentions.
----------------------------------------------------------------------------------------------------
                       total_count  neg_count  neg_rate
aspect                                                 
attitude-of-staff               56         44    0.7857
general-satisfaction           354        109    0.3079
speed                          100         30    0.3000
ease-of-use                     74         21    0.2838
price-value-for-money           40          8    0.2000
app-website                    253         46    0.1818

ANSWER: Staff attitude is Sable Row's biggest driver of dissatisfaction. It has the highest negative sentiment rate at 78.6% among all topics with at
least 30 mentions, ahead of general satisfaction (30.8%) and speed (30.0%).


In [41]:
df = ask("T7-H2")

segment = df[df.org == "Investa"]
summary = segment.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop orgs with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary = summary.sort_values(by="neg_rate", ascending=False)
print(summary)
print()
answer = (f"ANSWER: {ASPECT[summary.index[0]].capitalize()} is Investa's biggest driver of dissatisfaction. It has the "
          f"highest negative sentiment rate at {summary.neg_rate.iloc[0]:.1%} among all topics with at least {MIN_N} mentions, "
          f"ahead of {ASPECT[summary.index[1]]} ({summary.neg_rate.iloc[1]:.1%}) and {ASPECT[summary.index[2]]} "
          f"({summary.neg_rate.iloc[2]:.1%}).")
print(textwrap.fill(answer, width=150))

T7-H2  |  dataset: hard  |  answerable: yes
What is the biggest driver of dissatisfaction at Investa? Rank topics by negative sentiment rate and ignore any topics with less than 30 mentions.
----------------------------------------------------------------------------------------------------
                       total_count  neg_count  neg_rate
aspect                                                 
phone                           40         21    0.5250
price-value-for-money           34         14    0.4118
attitude-of-staff               54         17    0.3148
general-satisfaction            93         20    0.2151
app-website                    136         21    0.1544
ease-of-use                    155         18    0.1161

ANSWER: Phone support is Investa's biggest driver of dissatisfaction. It has the highest negative sentiment rate at 52.5% among all topics with at
least 30 mentions, ahead of price and value for money (41.2%) and staff attitude (31.5%).


In [42]:
# abstention: insufficient data
df = ask("T7-H3")

segment = df[df.org == "Pinecast"]
summary = segment.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop orgs with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
print(summary.sort_values(by="neg_rate", ascending=False))
print()
print("\nANSWER: No answer. All topics have insufficient reviews to claim that any of them drives dissatisfaction.")

T7-H3  |  dataset: hard  |  answerable: no (abstain)
What is the biggest driver of dissatisfaction at Pinecast? Rank topics by negative sentiment rate and ignore any topics with less than 30 mentions.
----------------------------------------------------------------------------------------------------
Empty DataFrame
Columns: [total_count, neg_count, neg_rate]
Index: []


ANSWER: No answer. All topics have insufficient reviews to claim that any of them drives dissatisfaction.


## T8 - Prioritisation by a named rule

In [43]:
df = ask("T8-E1")

segment = df[df.org == "Wayfare"]
total_count = len(segment) 
summary = segment.groupby("aspect").agg(mentions = ("sentiment", "size"), negative_mentions=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["negative_mentions"] >= MIN_N]

summary['prevalence'] = summary['mentions'] / total_count
summary['negative_rate'] = summary['negative_mentions'] / summary['mentions']
summary['priority_score'] = summary['prevalence'] * summary['negative_rate']
summary = summary.sort_values(by="priority_score", ascending=False)
print(summary)
print()

answer = (f"ANSWER: Wayfare should improve {ASPECT[summary.index[0]]} first (priority score: "
          f"{summary.priority_score.iloc[0]:.4f}), followed by {ASPECT[summary.index[1]]} (priority score: "
          f"{summary.priority_score.iloc[1]:.4f}).")
print(answer)

T8-E1  |  dataset: easy  |  answerable: yes
Which 2 issues should Wayfare address first? Rank aspects using this formula: priority score = aspect prevalence x aspect negative rate. Ignore any
aspects with less than 30 negative reviews.
----------------------------------------------------------------------------------------------------
                mentions  negative_mentions  prevalence  negative_rate  priority_score
aspect                                                                                
app-website          159                 82      0.2469         0.5157          0.1273
account-access        40                 37      0.0621         0.9250          0.0575
email                 40                 34      0.0621         0.8500          0.0528

ANSWER: Wayfare should improve the app or website first (priority score: 0.1273), followed by account access (priority score: 0.0575).


In [44]:
df = ask("T8-E2")

segment = df[df.org == "Larkmead Market"]
total_count = len(segment) 
summary = segment.groupby("aspect").agg(mentions = ("sentiment", "size"), negative_mentions=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["negative_mentions"] >= MIN_N]

summary['prevalence'] = summary['mentions'] / total_count
summary['negative_rate'] = summary['negative_mentions'] / summary['mentions']
summary['priority_score'] = summary['prevalence'] * summary['negative_rate']
summary = summary.sort_values(by="priority_score", ascending=False)
print(summary)
print()
answer = (f"ANSWER: Larkmead Market should improve {ASPECT[summary.index[0]]} first (priority score: "
          f"{summary.priority_score.iloc[0]:.4f}), followed by {ASPECT[summary.index[1]]} (priority score: "
          f"{summary.priority_score.iloc[1]:.4f}).")
print(answer)

T8-E2  |  dataset: easy  |  answerable: yes
Which 2 issues should Larkmead Market address first? Rank aspects using this formula: priority score = aspect prevalence x aspect negative rate.
Ignore any aspects with less than 30 negative reviews.
----------------------------------------------------------------------------------------------------
             mentions  negative_mentions  prevalence  negative_rate  priority_score
aspect                                                                             
app-website       259                124      0.2868         0.4788          0.1373
ease-of-use       175                108      0.1938         0.6171          0.1196
phone              40                 36      0.0443         0.9000          0.0399
reviews            40                 34      0.0443         0.8500          0.0377
email              40                 33      0.0443         0.8250          0.0365

ANSWER: Larkmead Market should improve the app or website first (p

In [45]:
df = ask("T8-H1")

segment = df[df.org == "Vella & Co"]
total_count = len(segment) 
summary = segment.groupby("aspect").agg(mentions = ("sentiment", "size"), negative_mentions=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["negative_mentions"] >= MIN_N]

summary['prevalence'] = summary['mentions'] / total_count
summary['negative_rate'] = summary['negative_mentions'] / summary['mentions']
summary['priority_score'] = summary['prevalence'] * summary['negative_rate']
summary = summary.sort_values(by="priority_score", ascending=False)
print(summary)
print()
answer = (f"ANSWER: Vella & Co should improve {ASPECT[summary.index[0]]} first (priority score: "
          f"{summary.priority_score.iloc[0]:.4f}), followed by {ASPECT[summary.index[1]]} (priority score: "
          f"{summary.priority_score.iloc[1]:.4f}).")
print(answer)

T8-H1  |  dataset: hard  |  answerable: yes
Which 2 issues should Vella & Co address first? Rank aspects using this formula: priority score = aspect prevalence x aspect negative rate. Ignore any
aspects with less than 30 negative reviews.
----------------------------------------------------------------------------------------------------
        mentions  negative_mentions  prevalence  negative_rate  priority_score
aspect                                                                        
speed        150                 44      0.1931         0.2933          0.0566
phone         40                 32      0.0515         0.8000          0.0412

ANSWER: Vella & Co should improve speed first (priority score: 0.0566), followed by phone support (priority score: 0.0412).


In [46]:
df = ask("T8-H2")

segment = df[df.org == "CompareHive"]
total_count = len(segment) 
summary = segment.groupby("aspect").agg(mentions = ("sentiment", "size"), negative_mentions=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["negative_mentions"] >= MIN_N]

summary['prevalence'] = summary['mentions'] / total_count
summary['negative_rate'] = summary['negative_mentions'] / summary['mentions']
summary['priority_score'] = summary['prevalence'] * summary['negative_rate']
summary = summary.sort_values(by="priority_score", ascending=False)
print(summary)
print()
answer = (f"ANSWER: CompareHive should improve {ASPECT[summary.index[0]]} first (priority score: "
          f"{summary.priority_score.iloc[0]:.4f}). The topic with the second highest priority score is "
          f"{ASPECT[summary.index[1]]}, which is not actionable.")
print(answer)

T8-H2  |  dataset: hard  |  answerable: yes
Which 2 issues should CompareHive address first? Rank aspects using this formula: priority score = aspect prevalence x aspect negative rate. Ignore
any aspects with less than 30 negative reviews.
----------------------------------------------------------------------------------------------------
                      mentions  negative_mentions  prevalence  negative_rate  priority_score
aspect                                                                                      
app-website                 80                 46      0.1027         0.5750          0.0591
general-satisfaction       116                 31      0.1489         0.2672          0.0398

ANSWER: CompareHive should improve the app or website first (priority score: 0.0591). The topic with the second highest priority score is general satisfaction, which is not actionable.


In [47]:
df = ask("T8-H3")

segment = df[df.org == "Kerbside"]
total_count = len(segment) 
summary = segment.groupby("aspect").agg(mentions = ("sentiment", "size"), negative_mentions=("sentiment", lambda x: (x == "negative").sum()))
summary = summary[summary["negative_mentions"] >= MIN_N]

summary['prevalence'] = summary['mentions'] / total_count
summary['negative_rate'] = summary['negative_mentions'] / summary['mentions']
summary['priority_score'] = summary['prevalence'] * summary['negative_rate']
summary = summary.sort_values(by="priority_score", ascending=False)
print(summary)
print()
answer = (f"ANSWER: Kerbside should improve {ASPECT[summary.index[0]]} first (priority score: "
          f"{summary.priority_score.iloc[0]:.4f}), followed by {ASPECT[summary.index[1]]} (priority score: "
          f"{summary.priority_score.iloc[1]:.4f}).")
print(answer)

T8-H3  |  dataset: hard  |  answerable: yes
Which 2 issues should Kerbside address first? Rank aspects using this formula: priority score = aspect prevalence x aspect negative rate. Ignore any
aspects with less than 30 negative reviews.
----------------------------------------------------------------------------------------------------
                      mentions  negative_mentions  prevalence  negative_rate  priority_score
aspect                                                                                      
app-website                 61                 32      0.2460         0.5246           0.129
email                       40                 31      0.1613         0.7750           0.125
general-satisfaction        51                 30      0.2056         0.5882           0.121

ANSWER: Kerbside should improve the app or website first (priority score: 0.1290), followed by email support (priority score: 0.1250).


## T9 - Comparative diagnosis and recommendation

In [48]:
df = ask("T9-E1")

seg_a = df[df.org == "Halden Savings"]
seg_b = df[df.org == "Kestrel Bank"]

# Each organisation broken down by topic and sentiment, then ranked by negative rate

# Halden Savings by topic and sentiment
summary_a = seg_a.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary_a = summary_a[summary_a["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary_a['neg_rate'] = summary_a['neg_count'] / summary_a['total_count']
summary_a = summary_a.sort_values(by="neg_rate", ascending=False)

# Kestrel Bank by topic and sentiment
summary_b = seg_b.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary_b = summary_b[summary_b["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary_b['neg_rate'] = summary_b['neg_count'] / summary_b['total_count']
summary_b = summary_b.sort_values(by="neg_rate", ascending=False)

top3_a = summary_a.head(3)
top3_b = summary_b.head(3)
print(f"Halden Savings top 3 topics by negative rate: {', '.join(top3_a.index)}")
print(f"Kestrel Bank top 3 topics by negative rate: {', '.join(top3_b.index)}")
print()

# For each of Halden Savings' top 3 topics, the gap against Kestrel Bank on the same topic
gaps = top3_a[["total_count", "neg_count", "neg_rate"]].copy()
gaps = gaps.rename(columns={"neg_rate": "neg_rate_a"})
gaps['neg_rate_b'] = summary_b['neg_rate']
gaps['gap_pp'] = ((gaps['neg_rate_a'] - gaps['neg_rate_b']) * 100).round(1)
gaps = gaps.sort_values(by="gap_pp", ascending=False)
print("Halden Savings top 3 topics vs Kestrel Bank on the same topic:")
print(gaps)

# The recommendation is the biggest gap on a topic the organisation can act on
fixable = gaps[~gaps.index.isin(NON_ACTIONABLE)]
print(f"\nLargest gap: {gaps.index[0]} ({gaps['gap_pp'].iloc[0]:+.1f} pp), largest actionable gap: {fixable.index[0]} ({fixable['gap_pp'].iloc[0]:+.1f} pp)")
print()
pick = fixable.index[0]
answer = (f"ANSWER: The higher complaint rate is driven by {ASPECT[pick]}, so Halden Savings should improve that first. "
          f"Reviews about {ASPECT[pick]} have a negative sentiment rate of {gaps.neg_rate_a[pick]:.1%}, compared with "
          f"{gaps.neg_rate_b[pick]:.1%} at Kestrel Bank - the largest gap between the two banks at "
          f"{gaps.gap_pp[pick]:.1f} percentage points.")
print(textwrap.fill(answer, width=150))

T9-E1  |  dataset: easy  |  answerable: yes
Why does Halden Savings have a higher complaint rate than Kestrel Bank? What should Halden Savings improve first?
----------------------------------------------------------------------------------------------------
Halden Savings top 3 topics by negative rate: attitude-of-staff, discounts-promotions, email
Kestrel Bank top 3 topics by negative rate: attitude-of-staff, email, phone

Halden Savings top 3 topics vs Kestrel Bank on the same topic:
                      total_count  neg_count  neg_rate_a  neg_rate_b  gap_pp
aspect                                                                      
discounts-promotions           40         33       0.825       0.600    22.5
email                          40         33       0.825       0.825     0.0
attitude-of-staff              40         35       0.875       0.950    -7.5

Largest gap: discounts-promotions (+22.5 pp), largest actionable gap: discounts-promotions (+22.5 pp)

ANSWER: The higher 

In [49]:
df = ask("T9-E2")

seg_a = df[df.org == "CompareHive"]
seg_b = df[df.org == "PricePilot"]

# Each organisation broken down by topic and sentiment, then ranked by negative rate

# CompareHive by topic and sentiment
summary_a = seg_a.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary_a = summary_a[summary_a["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary_a['neg_rate'] = summary_a['neg_count'] / summary_a['total_count']
summary_a = summary_a.sort_values(by="neg_rate", ascending=False)

# PricePilot by topic and sentiment
summary_b = seg_b.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary_b = summary_b[summary_b["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary_b['neg_rate'] = summary_b['neg_count'] / summary_b['total_count']
summary_b = summary_b.sort_values(by="neg_rate", ascending=False)

top3_a = summary_a.head(3)
top3_b = summary_b.head(3)
print(f"CompareHive top 3 topics by negative rate: {', '.join(top3_a.index)}")
print(f"PricePilot top 3 topics by negative rate: {', '.join(top3_b.index)}")
print()

# For each of CompareHive's top 3 topics, the gap against PricePilot on the same topic
gaps = top3_a[["total_count", "neg_count", "neg_rate"]].copy()
gaps = gaps.rename(columns={"neg_rate": "neg_rate_a"})
gaps['neg_rate_b'] = summary_b['neg_rate']
gaps['gap_pp'] = ((gaps['neg_rate_a'] - gaps['neg_rate_b']) * 100).round(1)
gaps = gaps.sort_values(by="gap_pp", ascending=False)
print("CompareHive top 3 topics vs PricePilot on the same topic:")
print(gaps)

# The recommendation is the biggest gap on a topic the organisation can act on
fixable = gaps[~gaps.index.isin(NON_ACTIONABLE)]
print(f"\nLargest gap: {gaps.index[0]} ({gaps['gap_pp'].iloc[0]:+.1f} pp), largest actionable gap: {fixable.index[0]} ({fixable['gap_pp'].iloc[0]:+.1f} pp)")
print()
pick = fixable.index[0]
answer = (f"ANSWER: The higher complaint rate is driven by {ASPECT[pick]}, so CompareHive should improve that first. "
          f"Reviews about {ASPECT[pick]} have a negative sentiment rate of {gaps.neg_rate_a[pick]:.1%}, compared with "
          f"{gaps.neg_rate_b[pick]:.1%} at PricePilot - the largest gap between the two price comparison platforms at "
          f"{gaps.gap_pp[pick]:.1f} percentage points.")
print(textwrap.fill(answer, width=150))

T9-E2  |  dataset: easy  |  answerable: yes
Why does CompareHive have a higher complaint rate than PricePilot? What should CompareHive improve first?
----------------------------------------------------------------------------------------------------
CompareHive top 3 topics by negative rate: account-access, app-website, discounts-promotions
PricePilot top 3 topics by negative rate: account-access, email, phone

CompareHive top 3 topics vs PricePilot on the same topic:
                      total_count  neg_count  neg_rate_a  neg_rate_b  gap_pp
aspect                                                                      
app-website                   100         74       0.740      0.3418    39.8
discounts-promotions           40         29       0.725      0.4000    32.5
account-access                 40         35       0.875      0.9250    -5.0

Largest gap: app-website (+39.8 pp), largest actionable gap: app-website (+39.8 pp)

ANSWER: The higher complaint rate is driven by the app 

In [50]:
df = ask("T9-H1")

seg_a = df[df.org == "Sable Row"]
seg_b = df[df.org == "Northerly"]

# Each organisation broken down by topic and sentiment, then ranked by negative rate

# Sable Row by topic and sentiment
summary_a = seg_a.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary_a = summary_a[summary_a["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary_a['neg_rate'] = summary_a['neg_count'] / summary_a['total_count']
summary_a = summary_a.sort_values(by="neg_rate", ascending=False)

# Northerly by topic and sentiment
summary_b = seg_b.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary_b = summary_b[summary_b["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary_b['neg_rate'] = summary_b['neg_count'] / summary_b['total_count']
summary_b = summary_b.sort_values(by="neg_rate", ascending=False)

top3_a = summary_a.head(3)
top3_b = summary_b.head(3)
print(f"Sable Row top 3 topics by negative rate: {', '.join(top3_a.index)}")
print(f"Northerly top 3 topics by negative rate: {', '.join(top3_b.index)}")
print()

# For each of Sable Row's top 3 topics, the gap against Northerly on the same topic
gaps = top3_a[["total_count", "neg_count", "neg_rate"]].copy()
gaps = gaps.rename(columns={"neg_rate": "neg_rate_a"})
gaps['neg_rate_b'] = summary_b['neg_rate']
gaps['gap_pp'] = ((gaps['neg_rate_a'] - gaps['neg_rate_b']) * 100).round(1)
gaps = gaps.sort_values(by="gap_pp", ascending=False)
print("Sable Row top 3 topics vs Northerly on the same topic:")
print(gaps)

# The recommendation is the biggest gap on a topic the organisation can act on
fixable = gaps[~gaps.index.isin(NON_ACTIONABLE)]
print(f"\nLargest gap: {gaps.index[0]} ({gaps['gap_pp'].iloc[0]:+.1f} pp), largest actionable gap: {fixable.index[0]} ({fixable['gap_pp'].iloc[0]:+.1f} pp)")
print()
pick = fixable.index[0]
answer = (f"ANSWER: The higher complaint rate is driven by {ASPECT[pick]}, so Sable Row should improve that first. "
          f"Reviews about {ASPECT[pick]} have a negative sentiment rate of {gaps.neg_rate_a[pick]:.1%}, compared with "
          f"{gaps.neg_rate_b[pick]:.1%} at Northerly - the largest actionable gap between the two fashion brands at "
          f"{gaps.gap_pp[pick]:.1f} percentage points. "
          f"{ASPECT[gaps.index[0]].capitalize()} has the largest gap overall, but it is not actionable.")
print(textwrap.fill(answer, width=150))

T9-H1  |  dataset: hard  |  answerable: yes
Why does Sable Row have a higher complaint rate than Northerly? What should Sable Row improve first?
----------------------------------------------------------------------------------------------------
Sable Row top 3 topics by negative rate: attitude-of-staff, general-satisfaction, speed
Northerly top 3 topics by negative rate: phone, attitude-of-staff, app-website

Sable Row top 3 topics vs Northerly on the same topic:
                      total_count  neg_count  neg_rate_a  neg_rate_b  gap_pp
aspect                                                                      
general-satisfaction          354        109      0.3079      0.0272    28.1
attitude-of-staff              56         44      0.7857      0.5476    23.8
speed                         100         30      0.3000      0.1511    14.9

Largest gap: general-satisfaction (+28.1 pp), largest actionable gap: attitude-of-staff (+23.8 pp)

ANSWER: The higher complaint rate is driven b

In [51]:
df = ask("T9-H2")

seg_a = df[df.org == "Larkmead Market"]
seg_b = df[df.org == "Oakpan Grocers"]

# Each organisation broken down by topic and sentiment, then ranked by negative rate

# Larkmead Market by topic and sentiment
summary_a = seg_a.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary_a = summary_a[summary_a["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary_a['neg_rate'] = summary_a['neg_count'] / summary_a['total_count']
summary_a = summary_a.sort_values(by="neg_rate", ascending=False)

# Oakpan Grocers by topic and sentiment
summary_b = seg_b.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary_b = summary_b[summary_b["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary_b['neg_rate'] = summary_b['neg_count'] / summary_b['total_count']
summary_b = summary_b.sort_values(by="neg_rate", ascending=False)

top3_a = summary_a.head(3)
top3_b = summary_b.head(3)
print(f"Larkmead Market top 3 topics by negative rate: {', '.join(top3_a.index)}")
print(f"Oakpan Grocers top 3 topics by negative rate: {', '.join(top3_b.index)}")
print()

# For each of Larkmead Market's top 3 topics, the gap against Oakpan Grocers on the same topic
gaps = top3_a[["total_count", "neg_count", "neg_rate"]].copy()
gaps = gaps.rename(columns={"neg_rate": "neg_rate_a"})
gaps['neg_rate_b'] = summary_b['neg_rate']
gaps['gap_pp'] = ((gaps['neg_rate_a'] - gaps['neg_rate_b']) * 100).round(1)
gaps = gaps.sort_values(by="gap_pp", ascending=False)
print("Larkmead Market top 3 topics vs Oakpan Grocers on the same topic:")
print(gaps)

# The recommendation is the biggest gap on a topic the organisation can act on
fixable = gaps[~gaps.index.isin(NON_ACTIONABLE)]
print(f"\nLargest gap: {gaps.index[0]} ({gaps['gap_pp'].iloc[0]:+.1f} pp), largest actionable gap: {fixable.index[0]} ({fixable['gap_pp'].iloc[0]:+.1f} pp)")
print()
pick = fixable.index[0]
answer = (f"ANSWER: The higher complaint rate is driven by {ASPECT[pick]}, so Larkmead Market should improve that first. "
          f"Reviews about {ASPECT[pick]} have a negative sentiment rate of {gaps.neg_rate_a[pick]:.1%}, compared with "
          f"{gaps.neg_rate_b[pick]:.1%} at Oakpan Grocers - the largest gap between the two grocery stores at "
          f"{gaps.gap_pp[pick]:.1f} percentage points.")
print(textwrap.fill(answer, width=150))

T9-H2  |  dataset: hard  |  answerable: yes
Why does Larkmead Market have a higher complaint rate than Oakpan Grocers? What should Larkmead Market improve first?
----------------------------------------------------------------------------------------------------
Larkmead Market top 3 topics by negative rate: ease-of-use, speed, app-website
Oakpan Grocers top 3 topics by negative rate: ease-of-use, discounts-promotions, attitude-of-staff

Larkmead Market top 3 topics vs Oakpan Grocers on the same topic:
             total_count  neg_count  neg_rate_a  neg_rate_b  gap_pp
aspect                                                             
speed                 51         29      0.5686      0.2368    33.2
ease-of-use          170         98      0.5765      0.4085    16.8
app-website          189         87      0.4603         NaN     NaN

Largest gap: speed (+33.2 pp), largest actionable gap: speed (+33.2 pp)

ANSWER: The higher complaint rate is driven by speed, so Larkmead Market shoul

In [52]:
df = ask("T9-H3")

seg_a = df[df.org == "Halden Savings"]
seg_b = df[df.org == "Northeast Bank"]

# Each organisation broken down by topic and sentiment, then ranked by negative rate

# Halden Savings by topic and sentiment
summary_a = seg_a.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary_a = summary_a[summary_a["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary_a['neg_rate'] = summary_a['neg_count'] / summary_a['total_count']
summary_a = summary_a.sort_values(by="neg_rate", ascending=False)

# Northeast Bank by topic and sentiment
summary_b = seg_b.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary_b = summary_b[summary_b["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary_b['neg_rate'] = summary_b['neg_count'] / summary_b['total_count']
summary_b = summary_b.sort_values(by="neg_rate", ascending=False)

top3_a = summary_a.head(3)
top3_b = summary_b.head(3)
print(f"Halden Savings top 3 topics by negative rate: {', '.join(top3_a.index)}")
print(f"Northeast Bank top 3 topics by negative rate: {', '.join(top3_b.index)}")
print()

# For each of Halden Savings' top 3 topics, the gap against Northeast Bank on the same topic
gaps = top3_a[["total_count", "neg_count", "neg_rate"]].copy()
gaps = gaps.rename(columns={"neg_rate": "neg_rate_a"})
gaps['neg_rate_b'] = summary_b['neg_rate']
gaps['gap_pp'] = ((gaps['neg_rate_a'] - gaps['neg_rate_b']) * 100).round(1)
gaps = gaps.sort_values(by="gap_pp", ascending=False)
print("Halden Savings top 3 topics vs Northeast Bank on the same topic:")
print(gaps)

# The recommendation is the biggest gap on a topic the organisation can act on
fixable = gaps[~gaps.index.isin(NON_ACTIONABLE)]
print(f"\nLargest gap: {gaps.index[0]} ({gaps['gap_pp'].iloc[0]:+.1f} pp), largest actionable gap: {fixable.index[0]} ({fixable['gap_pp'].iloc[0]:+.1f} pp)")
print()
pick = fixable.index[0]
answer = (f"ANSWER: No answer. The data does not explain what drives the higher complaint rate. The only topic where Halden Savings "
          f"has a higher negative rate than Northeast Bank is {ASPECT[gaps.index[0]]}, but it is not actionable. Halden Savings should collect more "
          f"customer feedback to understand what is driving dissatisfaction.")
print(textwrap.fill(answer, width=150))

T9-H3  |  dataset: hard  |  answerable: yes
Why does Halden Savings have a higher complaint rate than Northeast Bank? What should Halden Savings improve first?
----------------------------------------------------------------------------------------------------
Halden Savings top 3 topics by negative rate: general-satisfaction, ease-of-use, speed
Northeast Bank top 3 topics by negative rate: ease-of-use, speed, app-website

Halden Savings top 3 topics vs Northeast Bank on the same topic:
                      total_count  neg_count  neg_rate_a  neg_rate_b  gap_pp
aspect                                                                      
general-satisfaction           80         31      0.3875      0.1359    25.2
ease-of-use                   107         28      0.2617      0.3333    -7.2
speed                          40          6      0.1500      0.2500   -10.0

Largest gap: general-satisfaction (+25.2 pp), largest actionable gap: ease-of-use (-7.2 pp)

ANSWER: No answer. The data d

## T10 - Full CX report

In [53]:
df = ask("T10-E1")

segment = df[df.org == "Trippa"]
industry = df[df.industry == "Travel Booking"]

# 1. Overall sentiment, as shares of all Trippa reviews
mix = segment["sentiment"].value_counts().to_frame("total_count")
mix['share'] = mix['total_count'] / len(segment)
print(f"Overall sentiment - Trippa (Travel Booking), {len(segment)} reviews")
print(mix)
print()

# 2. Trippa by topic and sentiment, ranked by negative rate. Account access and email support
#    both have 35 negatives out of 40 mentions, so they are joint first: the rank column keeps
#    that visible instead of letting the sort order invent a winner.
summary = segment.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary['rank'] = summary['neg_rate'].rank(method="min", ascending=False).astype(int)
summary = summary.sort_values(by="rank", kind="stable")
top3 = summary[summary["rank"] <= 3]
print("Top 3 issues by negative sentiment rate:")
print(top3)
print()

# 3. The joint top issues against the industry. The industry slice contains Trippa's own
#    reviews, so the two samples would not be independent: Trippa is removed from the
#    industry totals and the z-test compares it with the rest of Travel Booking. The tie
#    means there are two top issues to test, not one.
tied = top3[top3["rank"] == 1]
for top in tied.index:
    seg_ind = industry[industry.aspect == top]
    total_ind = len(seg_ind)
    neg_ind = (seg_ind["sentiment"] == "negative").sum()

    total_org = top3.loc[top, "total_count"]
    neg_org = top3.loc[top, "neg_count"]
    rate_org = top3.loc[top, "neg_rate"]

    total_rest = total_ind - total_org
    neg_rest = neg_ind - neg_org
    rate_rest = neg_rest / total_rest

    z_stat, p_value = proportions_ztest(count=[neg_org, neg_rest], nobs=[total_org, total_rest], alternative="two-sided")
    print(f"Joint top issue vs industry - {top}")
    print(f"Trippa: {rate_org:.1%} negative (n = {total_org})")
    print(f"Rest of Travel Booking: {rate_rest:.1%} negative (n = {total_rest})")
    print(f"Gap vs the rest of the industry: {(rate_org - rate_rest) * 100:+.1f} pp, z = {z_stat:.3f}, p = {p_value:.4f}")
    print()

# 4. The roadmap follows the driver ranking, minus any topic Trippa cannot act on. The two
#    joint-first topics share the first step.
roadmap = top3[~top3.index.isin(NON_ACTIONABLE)]
print("Roadmap order (driver ranking):")
for r in sorted(roadmap['rank'].unique()):
    print(f"  rank {r}: {', '.join(roadmap.index[roadmap['rank'] == r])}")
print()
answer = """ANSWER: CX report for Trippa

  Overview: Trippa received 674 labelled mentions - 56.2% positive, 42.3% negative, 1.5% neutral.

  Top 3 issues:
    1. account access (87.5% negative, 40 mentions) - joint first
    1. email support (87.5% negative, 40 mentions) - joint first
    3. phone support (65.0% negative, 40 mentions)

  Industry comparison: Trippa performs better than the Travel Booking average on account access (87.5% vs 93.1%) and slightly worse on email support (87.5% vs 86.2%). Tested against the rest of the industry, neither difference is statistically significant (p = 0.104 for account access, p = 0.791 for email support), so Trippa's performance is broadly similar to its industry peers.
  
  Roadmap:
    1. Improve account access and email support together - tied on both mentions and negative rate
    2. Improve phone support (65.0% negative across 40 mentions)"""
# the report is a structured list, so each line is wrapped on its own: filling the whole
# string as one paragraph would collapse the line breaks and split the numbering
for line in answer.splitlines():
    print(textwrap.fill(line, width=150, subsequent_indent="    "))

T10-E1  |  dataset: easy  |  answerable: yes
Write a CX report for Trippa. Cover overall sentiment, the top 3 issues ranked by negative sentiment rate, and how the top issue compares with the
industry average using a statistical significance test. Also give me an improvement roadmap.
----------------------------------------------------------------------------------------------------
Overall sentiment - Trippa (Travel Booking), 674 reviews
           total_count   share
sentiment                     
positive           379  0.5623
negative           285  0.4228
neutral             10  0.0148

Top 3 issues by negative sentiment rate:
                total_count  neg_count  neu_count  pos_count  neg_rate  rank
aspect                                                                      
account-access           40         35          2          3     0.875     1
email                    40         35          0          5     0.875     1
phone                    40         26          3   

In [54]:
df = ask("T10-E2")

segment = df[df.org == "Kestrel Bank"]
industry = df[df.industry == "Banking"]

# 1. Overall sentiment, as shares of all Kestrel Bank reviews
mix = segment["sentiment"].value_counts().to_frame("total_count")
mix['share'] = mix['total_count'] / len(segment)
print(f"Overall sentiment - Kestrel Bank (Banking), {len(segment)} reviews")
print(mix)
print()

# 2. Kestrel Bank by topic and sentiment, ranked by negative rate. Email and phone support
#    both have 33 negatives out of 40 mentions, so they are joint second: the rank column
#    keeps that visible instead of letting the sort order invent an order between them.
summary = segment.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary['rank'] = summary['neg_rate'].rank(method="min", ascending=False).astype(int)
summary = summary.sort_values(by="rank", kind="stable")
top3 = summary[summary["rank"] <= 3]
print("Top 3 issues by negative sentiment rate:")
print(top3)
print()

# 3. The top issue against the industry. The industry slice contains Kestrel Bank's own
#    reviews, so the two samples would not be independent: Kestrel Bank is removed from the
#    industry totals and the z-test compares it with the rest of Banking. Rank 1 is untied
#    here, so the loop runs once.
tied = top3[top3["rank"] == 1]
for top in tied.index:
    seg_ind = industry[industry.aspect == top]
    total_ind = len(seg_ind)
    neg_ind = (seg_ind["sentiment"] == "negative").sum()
    rate_ind = neg_ind / total_ind

    total_org = top3.loc[top, "total_count"]
    neg_org = top3.loc[top, "neg_count"]
    rate_org = top3.loc[top, "neg_rate"]

    total_rest = total_ind - total_org
    neg_rest = neg_ind - neg_org
    rate_rest = neg_rest / total_rest

    z_stat, p_value = proportions_ztest(count=[neg_org, neg_rest], nobs=[total_org, total_rest], alternative="two-sided")
    print(f"Top issue vs industry - {top}")
    print(f"Kestrel Bank: {rate_org:.1%} negative (n = {total_org})")
    print(f"Banking average: {rate_ind:.1%} negative (n = {total_ind}, Kestrel Bank included)")
    print(f"Rest of Banking: {rate_rest:.1%} negative (n = {total_rest})")
    print(f"Gap vs the rest of the industry: {(rate_org - rate_rest) * 100:+.1f} pp, z = {z_stat:.3f}, p = {p_value:.4f}")
    print()

# 4. The roadmap follows the driver ranking, minus any topic Kestrel Bank cannot act on. The
#    two joint-second topics share a step.
roadmap = top3[~top3.index.isin(NON_ACTIONABLE)]
print("Roadmap order (driver ranking):")
for r in sorted(roadmap['rank'].unique()):
    print(f"  rank {r}: {', '.join(roadmap.index[roadmap['rank'] == r])}")
print()
answer = """ANSWER: CX report for Kestrel Bank

  Overview: Kestrel Bank received 709 labelled mentions - 51.5% positive, 38.9% negative, 9.6% neutral.

  Top 3 issues:
    1. staff attitude (95.0% negative, 40 mentions)
    2. email support (82.5% negative, 40 mentions) - joint second
    2. phone support (82.5% negative, 40 mentions) - joint second

  Industry comparison: Kestrel Bank performs worse than the Banking average on staff attitude (95.0% vs 90.0%). Tested against the rest of the industry, the difference is not statistically significant (p = 0.224), so Kestrel Bank's performance is broadly similar to its industry peers.

  Roadmap:
    1. Improve staff attitude (95.0% negative across 40 mentions)
    2. Improve email support and phone support together - tied on both mentions and negative rate"""

for line in answer.splitlines():
    print(textwrap.fill(line, width=150, subsequent_indent="    "))

T10-E2  |  dataset: easy  |  answerable: yes
Write a CX report for Kestrel Bank. Cover overall sentiment, the top 3 issues ranked by negative sentiment rate, and how the top issue compares with
the industry average using a statistical significance test. Also give me an improvement roadmap.
----------------------------------------------------------------------------------------------------
Overall sentiment - Kestrel Bank (Banking), 709 reviews
           total_count   share
sentiment                     
positive           365  0.5148
negative           276  0.3893
neutral             68  0.0959

Top 3 issues by negative sentiment rate:
                   total_count  neg_count  neu_count  pos_count  neg_rate  rank
aspect                                                                         
attitude-of-staff           40         38          0          2     0.950     1
email                       40         33          0          7     0.825     2
phone                       40     

In [55]:
df = ask("T10-H1")

segment = df[df.org == "Vella & Co"]
industry = df[df.industry == "Fashion"]

# 1. Overall sentiment, as shares of all Vella & Co reviews
mix = segment["sentiment"].value_counts().to_frame("total_count")
mix['share'] = mix['total_count'] / len(segment)
print(f"Overall sentiment - Vella & Co (Fashion), {len(segment)} reviews")
print(mix)
print()

# 2. Vella & Co by topic and sentiment, ranked by negative rate
summary = segment.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary = summary.sort_values(by="neg_rate", ascending=False)
top3 = summary.head(3)
print("Top 3 issues by negative sentiment rate:")
print(top3)
print()

# 3. The top pain point against the industry. The industry slice contains Vella & Co's own
#    reviews, so the two samples would not be independent: Vella & Co is removed from the
#    industry totals and the z-test compares it with the rest of Fashion.
top = top3.index[0]
seg_ind = industry[industry.aspect == top]
total_ind = len(seg_ind)
neg_ind = (seg_ind["sentiment"] == "negative").sum()
rate_ind = neg_ind / total_ind

total_org = top3.loc[top, "total_count"]
neg_org = top3.loc[top, "neg_count"]
rate_org = top3.loc[top, "neg_rate"]

total_rest = total_ind - total_org
neg_rest = neg_ind - neg_org
rate_rest = neg_rest / total_rest

z_stat, p_value = proportions_ztest(count=[neg_org, neg_rest], nobs=[total_org, total_rest], alternative="two-sided")
print(f"Top issue vs industry - {top}")
print(f"Vella & Co: {rate_org:.1%} negative (n = {total_org})")
print(f"Fashion average: {rate_ind:.1%} negative (n = {total_ind}, Vella & Co included)")
print(f"Rest of Fashion: {rate_rest:.1%} negative (n = {total_rest})")
print(f"Gap vs the rest of the industry: {(rate_org - rate_rest) * 100:+.1f} pp, z = {z_stat:.3f}, p = {p_value:.4f}")
print()

# 4. The roadmap follows the driver ranking, minus any topic Vella & Co cannot act on
roadmap = top3[~top3.index.isin(NON_ACTIONABLE)]
print(f"Roadmap order (driver ranking): {', '.join(roadmap.index)}")
print()
answer = """ANSWER: CX report for Vella & Co

  Overview: Vella & Co received 777 labelled mentions - 74.5% positive, 24.3% negative, 1.2% neutral.

  Top 3 issues:
    1. phone support (80.0% negative, 40 mentions)
    2. staff attitude (48.3% negative, 58 mentions)
    3. price and value for money (41.7% negative, 36 mentions)

  Industry comparison: Vella & Co performs worse than the Fashion average on phone support (80.0% vs 74.5%). Tested against the rest of the industry, the difference is not statistically significant (p = 0.290), so Vella & Co's performance is broadly similar to its industry peers.

  Roadmap:
    1. Improve phone support (80.0% negative across 40 mentions)
    2. Improve staff attitude (48.3% negative across 58 mentions)
    3. Improve price and value for money (41.7% negative across 36 mentions)"""

for line in answer.splitlines():
    print(textwrap.fill(line, width=150, subsequent_indent="    "))

T10-H1  |  dataset: hard  |  answerable: yes
Write a CX report for Vella & Co. Cover overall sentiment, the top 3 issues ranked by negative sentiment rate, and how the top issue compares with the
industry average using a statistical significance test. Also give me an improvement roadmap.
----------------------------------------------------------------------------------------------------
Overall sentiment - Vella & Co (Fashion), 777 reviews
           total_count   share
sentiment                     
positive           579  0.7452
negative           189  0.2432
neutral              9  0.0116

Top 3 issues by negative sentiment rate:
                       total_count  neg_count  neu_count  pos_count  neg_rate
aspect                                                                       
phone                           40         32          3          5    0.8000
attitude-of-staff               58         28          1         29    0.4828
price-value-for-money           36         15  

In [56]:
df = ask("T10-H2")

segment = df[df.org == "Freshbury"]
industry = df[df.industry == "Groceries"]

# 1. Overall sentiment, as shares of all Freshbury reviews
mix = segment["sentiment"].value_counts().to_frame("total_count")
mix['share'] = mix['total_count'] / len(segment)
print(f"Overall sentiment - Freshbury (Groceries), {len(segment)} reviews")
print(mix)
print()

# 2. Freshbury by topic and sentiment, ranked by negative rate
summary = segment.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
summary = summary[summary["total_count"] >= MIN_N]  # drop topics with < 30 reviews
summary['neg_rate'] = summary['neg_count'] / summary['total_count']
summary = summary.sort_values(by="neg_rate", ascending=False)
top3 = summary.head(3)
print("Top 3 issues by negative sentiment rate:")
print(top3)
print()

# 3. The top pain point against the industry. The industry slice contains Freshbury's own
#    reviews, so the two samples would not be independent: Freshbury is removed from the
#    industry totals and the z-test compares it with the rest of Groceries.
top = top3.index[0]
seg_ind = industry[industry.aspect == top]
total_ind = len(seg_ind)
neg_ind = (seg_ind["sentiment"] == "negative").sum()
rate_ind = neg_ind / total_ind

total_org = top3.loc[top, "total_count"]
neg_org = top3.loc[top, "neg_count"]
rate_org = top3.loc[top, "neg_rate"]

total_rest = total_ind - total_org
neg_rest = neg_ind - neg_org
rate_rest = neg_rest / total_rest

z_stat, p_value = proportions_ztest(count=[neg_org, neg_rest], nobs=[total_org, total_rest], alternative="two-sided")
print(f"Top issue vs industry - {top}")
print(f"Freshbury: {rate_org:.1%} negative (n = {total_org})")
print(f"Groceries average: {rate_ind:.1%} negative (n = {total_ind}, Freshbury included)")
print(f"Rest of Groceries: {rate_rest:.1%} negative (n = {total_rest})")
print(f"Gap vs the rest of the industry: {(rate_org - rate_rest) * 100:+.1f} pp, z = {z_stat:.3f}, p = {p_value:.4f}")
print()

# 4. The roadmap follows the driver ranking, minus any topic Freshbury cannot act on
roadmap = top3[~top3.index.isin(NON_ACTIONABLE)]
print(f"Roadmap order (driver ranking): {', '.join(roadmap.index)}")
print()
answer = """ANSWER: CX report for Freshbury

  Overview: Freshbury received 677 labelled mentions - 59.2% positive, 38.8% negative, 1.9% neutral.

  Top 3 issues:
    1. the app or website (49.6% negative, 262 mentions)
    2. speed (38.9% negative, 54 mentions)
    3. ease of use (30.2% negative, 189 mentions)

  Industry comparison: Freshbury performs worse than the Groceries average on the app or website (49.6% vs 48.1%). Tested against the rest of the industry, the difference is not statistically significant (p = 0.452), so Freshbury's performance is broadly similar to its industry peers.

  Roadmap:
    1. Improve the app or website (49.6% negative across 262 mentions)
    2. Improve speed (38.9% negative across 54 mentions)
    3. Improve ease of use (30.2% negative across 189 mentions)"""

for line in answer.splitlines():
    print(textwrap.fill(line, width=150, subsequent_indent="    "))

T10-H2  |  dataset: hard  |  answerable: yes
Write a CX report for Freshbury. Cover overall sentiment, the top 3 issues ranked by negative sentiment rate, and how the top issue compares with the
industry average using a statistical significance test. Also give me an improvement roadmap.
----------------------------------------------------------------------------------------------------
Overall sentiment - Freshbury (Groceries), 677 reviews
           total_count   share
sentiment                     
positive           401  0.5923
negative           263  0.3885
neutral             13  0.0192

Top 3 issues by negative sentiment rate:
             total_count  neg_count  neu_count  pos_count  neg_rate
aspect                                                             
app-website          262        130          5        127    0.4962
speed                 54         21          0         33    0.3889
ease-of-use          189         57          0        132    0.3016

Top issue vs indus

In [57]:
# abstention: only one topic has 30 reviews, so there is no top 3 and no roadmap
df = ask("T10-H3")

segment = df[df.org == "Lumora"]
industry = df[df.industry == "Streaming"]

# 1. Overall sentiment, as shares of all Lumora reviews - this component is computable
mix = segment["sentiment"].value_counts().to_frame("total_count")
mix['share'] = mix['total_count'] / len(segment)
print(f"Overall sentiment - Lumora (Streaming), {len(segment)} reviews")
print(mix)
print()

# 2. Lumora by topic and sentiment: the floor leaves too few topics to rank
all_topics = segment.groupby("aspect").agg(total_count = ("sentiment", "size"), neg_count=("sentiment", lambda x: (x == "negative").sum()), neu_count=("sentiment", lambda x: (x == "neutral").sum()), pos_count=("sentiment", lambda x: (x == "positive").sum()))
all_topics['neg_rate'] = all_topics['neg_count'] / all_topics['total_count']
all_topics = all_topics.sort_values(by="neg_rate", ascending=False)
print(f"Topics covered: {len(all_topics)} of 12")
print(all_topics)
print()

summary = all_topics[all_topics["total_count"] >= MIN_N]  # drop topics with < 30 reviews
print(f"Topics clearing the {MIN_N}-review floor: {len(summary)} ({', '.join(summary.index)})")
print()

# 3. Industry comparison. The industry slice contains Lumora's own reviews, so Lumora is
#    removed from the industry totals: a two-proportion z-test needs independent samples,
#    and the comparison group is the rest of Streaming.
top = summary.index[0]
seg_ind = industry[industry.aspect == top]
total_ind = len(seg_ind)
neg_ind = (seg_ind["sentiment"] == "negative").sum()
rate_ind = neg_ind / total_ind

total_org = summary.loc[top, "total_count"]
neg_org = summary.loc[top, "neg_count"]
rate_org = summary.loc[top, "neg_rate"]

total_rest = total_ind - total_org
neg_rest = neg_ind - neg_org
rate_rest = neg_rest / total_rest

z_stat, p_value = proportions_ztest(count=[neg_org, neg_rest], nobs=[total_org, total_rest], alternative="two-sided")
print(f"Only eligible topic vs industry - {top}")
print(f"Lumora: {rate_org:.1%} negative (n = {total_org})")
print(f"Streaming average: {rate_ind:.1%} negative (n = {total_ind}, Lumora included)")
print(f"Rest of Streaming: {rate_rest:.1%} negative (n = {total_rest})")
print(f"Gap vs the rest of the industry: {(rate_org - rate_rest) * 100:+.1f} pp, z = {z_stat:.3f}, p = {p_value:.4f}")
print()

answer = ("ANSWER: No answer. A full CX report cannot be produced because there is insufficient data to rank the top 3 issues and create a roadmap. " \
"Only one topic (account access) has at least 30 reviews. The negative sentiment rate is 35%, which is slightly lower than the 36.7% for the rest of the industry, and the difference is not statistically significant (p=0.8653). ")
print(textwrap.fill(answer, width=150))

T10-H3  |  dataset: hard  |  answerable: no (abstain)
Write a CX report for Lumora. Cover overall sentiment, the top 3 issues ranked by negative sentiment rate, and how the top issue compares with the
industry average using a statistical significance test. Also give me an improvement roadmap.
----------------------------------------------------------------------------------------------------
Overall sentiment - Lumora (Streaming), 56 reviews
           total_count   share
sentiment                     
neutral             32  0.5714
negative            18  0.3214
positive             6  0.1071

Topics covered: 3 of 12
                      total_count  neg_count  neu_count  pos_count  neg_rate
aspect                                                                      
account-access                 40         14         25          1    0.3500
app-website                    12          4          6          2    0.3333
discounts-promotions            4          0          1          3

# End of notebook